In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd

os.environ['OPENAI_KEY_SAE'] = '...' # Replace with your OpenAI API key, or with another environment variable (e.g. os.environ['OPENAI_API_KEY'])
os.environ['OPENAI_BASE_URL'] = 'http://0.0.0.0:8001/v1'  # Optional: local OpenAI-compatible endpoint
# os.environ['OPENAI_BASE_URL'] = 'http://0.0.0.0:8010/v1'  # Optional: local OpenAI-compatible endpoint

from hypothesaes.quickstart import train_sae

from hypothesaes.embedding import get_local_embeddings
from hypothesaes.sae import get_multiple_sae_activations
from hypothesaes.select_neurons import select_neurons
from hypothesaes.interpret_neurons import NeuronInterpreter, InterpretConfig, ScoringConfig, SamplingConfig, sample_percentile_bins
from hypothesaes.annotate import annotate_texts_with_concepts
from hypothesaes.evaluation import score_hypotheses, compute_ols_metrics

current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    prefix = "../"
else:
    prefix = "./"
base_dir = os.path.join(prefix, "data")

## Headlines

Note that Headlines requires some special handling because it is a pairwise dataset.  
Each example contains two headlines, A and B, and a binary label: 1 if headline A received higher clicks than B, 0 otherwise. A and B were randomly shuffled, so the label is approximately balanced.

We apply HypotheSAEs as follows:
1. Train SAE on all unique headlines in the train set.
2. Compute $Z_A$, an $N \times M$ activation matrix for all headline A's; similarly compute $Z_B$.
3. Compute $Z_{\Delta} = Z_A - Z_B$, which represents how much more (or less) each feature activates in headline A compared to B.
4. Select 20 neurons using L1 logistic regression on $Z_{\Delta}$.
5. Interpret selected neurons by sampling from the texts originally used to train the SAE.

For evaluation:
1. Compute $H_A$, an $N \times 20$ binary matrix of hypothesis annotations for all headlines A, and $H_B$ similarly.
2. Compute $H_{\Delta} = H_A - H_B$, and regress the label on this difference. Note that $H_{\Delta}$ can take on values in $\{-1, 0, 1\}$, which we account for in `src.evaluation.compute_hypothesis_separation_scores`.


In [4]:
DATASET_NAME = "headlines"
# EMBEDDER = "text-embedding-3-small"
# text-embedding-3-small is closed source; using this one for now
EMBEDDER = "nomic-ai/modernbert-embed-base"
CACHE_NAME = f"{DATASET_NAME}_{EMBEDDER}"

label_col = "label_pairwise"
train_df = pd.read_json(os.path.join(base_dir, DATASET_NAME, "train-8.8K.json"), lines=True)
val_df = pd.read_json(os.path.join(base_dir, DATASET_NAME, "val-1.0K.json"), lines=True)

train_labels = train_df[label_col].values

unique_train_texts = np.unique(train_df["headline_A"].tolist() + train_df["headline_B"].tolist()).tolist()
unique_val_texts = np.unique(val_df["headline_A"].tolist() + val_df["headline_B"].tolist()).tolist()

# text2embedding = get_openai_embeddings(unique_train_texts + unique_val_texts, model=EMBEDDER, cache_name=CACHE_NAME)
text2embedding = get_local_embeddings(unique_train_texts + unique_val_texts, model=EMBEDDER, cache_name=CACHE_NAME)

train_embeddings = np.stack([text2embedding[text] for text in unique_train_texts])
val_embeddings = np.stack([text2embedding[text] for text in unique_val_texts])

Loading embedding chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Loaded 16468 embeddings in 0.2s


In [5]:
checkpoint_dir = f"./checkpoints/{CACHE_NAME}"
sae_256_8 = train_sae(embeddings=train_embeddings, M=256, K=8, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=200, patience=5)
sae_32_4 = train_sae(embeddings=train_embeddings, M=32, K=4, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=200, patience=5)

sae_list = [sae_256_8, sae_32_4]

unique_train_activations, neuron_source_sae_info = get_multiple_sae_activations(sae_list, train_embeddings, return_neuron_source_info=True)

headline_A_train = np.stack([text2embedding[text] for text in train_df["headline_A"].tolist()])
headline_B_train = np.stack([text2embedding[text] for text in train_df["headline_B"].tolist()])

Z_A = get_multiple_sae_activations(sae_list, headline_A_train)
Z_B = get_multiple_sae_activations(sae_list, headline_B_train)
Z_delta = Z_A - Z_B

Loaded model from ./checkpoints/headlines_nomic-ai/modernbert-embed-base/SAE_M=256_K=8.pt onto device cpu
Loaded model from ./checkpoints/headlines_nomic-ai/modernbert-embed-base/SAE_M=32_K=4.pt onto device cpu


Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
selection_cache_dir = os.path.join(prefix, "selection_cache", CACHE_NAME)
os.makedirs(selection_cache_dir, exist_ok=True)
selected_neurons, scores = select_neurons(
    activations=Z_delta,
    target=train_labels,
    n_select=20,
    method="lasso",
    classification=True,
    verbose=True,
    cache_dir=selection_cache_dir
)

In [7]:
ANNOTATOR_MODEL = "openai/gpt-oss-20b"

In [8]:
interpreter = NeuronInterpreter(
    interpreter_model="openai/gpt-oss-120b",
    annotator_model=ANNOTATOR_MODEL,
    n_workers_interpretation=4,
    n_workers_annotation=4,
    cache_name=CACHE_NAME,
)

interpretation_sampling_config = SamplingConfig(
    function=sample_percentile_bins,
    n_examples=20,
    sampling_kwargs={
        "high_percentile": (80, 100),
        "low_percentile": None, # Randomly samples from zero-activating examples
    },
)

interpret_config = InterpretConfig(
    sampling=interpretation_sampling_config,
    n_candidates=5,
    task_specific_instructions="""All of the texts are digital news headlines.
Features should describe specific characteristics in the headline texts. Example features may include:
- "explicitly mentions climate change or another environmental issue"
- "asks the reader a rhetorical question\"""",
)

In [9]:
interpretations = interpreter.interpret_neurons(
    texts=unique_train_texts,
    activations=unique_train_activations,
    neuron_indices=selected_neurons,
    config=interpret_config,
)

In [10]:
scoring_config = ScoringConfig(
    n_examples=200,
    sampling_function=sample_percentile_bins,
    sampling_kwargs={
        "high_percentile": (80, 100),
        "low_percentile": None, # Randomly samples from zero-activating examples
    },
)

all_metrics = interpreter.score_interpretations(
    texts=unique_train_texts,
    activations=unique_train_activations,
    interpretations=interpretations,
    config=scoring_config,
)


[WARNING] There are less than 100 examples in bin (80, 100) for neuron 146; using 81 instead
Found 19905 cached items; annotating 0 uncached items


In [13]:
results = []
for neuron_idx, fidelity_scores in all_metrics.items():
    best_interp = max(interpretations[neuron_idx], key=lambda interp: fidelity_scores[interp]['f1'])
    results.append({
        'neuron_idx': neuron_idx,
        'source_sae': neuron_source_sae_info[neuron_idx],
        'target_lasso_coef': scores[selected_neurons.index(neuron_idx)],
        'interpretation': best_interp,
        'f1_fidelity_score': fidelity_scores[best_interp]['f1'],
    })

hypothesis_df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
display(hypothesis_df.sort_values(by='target_lasso_coef', ascending=False))
pd.reset_option('display.max_colwidth')

,neuron_idx,source_sae,target_lasso_coef,interpretation,f1_fidelity_score
1,266,"(SAE_1, 32, 4)",0.136803,mentions a video or footage,0.959184
3,256,"(SAE_1, 32, 4)",0.054724,"includes a reference to visual media such as photos, pictures, images, graphics, or a photographer",0.879121
4,151,"(SAE_0, 256, 8)",0.049171,contains a standalone female pronoun ('She' or 'Her') as an individual word,0.995025
5,187,"(SAE_0, 256, 8)",0.034277,"contains a third‑person pronoun (He, They, Their) directly followed by a past‑tense verb (including irregular forms or negative contractions)",0.756098
6,14,"(SAE_0, 256, 8)",0.027620,"mentions the word \""People\",0.751515
8,13,"(SAE_0, 256, 8)",0.019672,"mentions a form of artistic or cultural work such as a song, poem, rap, talk, parody, anthem, or toy",0.745763
9,268,"(SAE_1, 32, 4)",0.015934,"contains a verb that describes speaking or asking (e.g., talk, say, tell, explain, respond, ask)",0.865979
10,82,"(SAE_0, 256, 8)",0.014938,"mentions a singular child (e.g., 'Kid', 'Boy', or 'Little Girl') in the headline",0.964824
11,132,"(SAE_0, 256, 8)",0.014375,"mentions a racial identifier such as the words 'Black', 'White', or the term 'Race'",0.984925
12,146,"(SAE_0, 256, 8)",0.011643,"contains the word “Guy”, “Man”, or “Dude”",0.955975


In [ ]:
train_delta_annots = {}

In [14]:
holdout_df = pd.read_json(os.path.join(base_dir, DATASET_NAME, "holdout-4.4K.json"), lines=True)

headline_A_holdout = holdout_df["headline_A"].tolist()
headline_B_holdout = holdout_df["headline_B"].tolist()
labels_holdout = holdout_df[label_col].values

hypotheses = hypothesis_df["interpretation"].tolist()

In [15]:
headline_A_annotations = annotate_texts_with_concepts(
    texts=headline_A_holdout,
    concepts=hypotheses,
    cache_name=CACHE_NAME,
    model=ANNOTATOR_MODEL,
    n_workers=100
)

Found 87140 cached items; annotating 0 uncached items


In [16]:

headline_B_annotations = annotate_texts_with_concepts(
    texts=headline_B_holdout,
    concepts=hypotheses,
    cache_name=CACHE_NAME,
    model=ANNOTATOR_MODEL,
    n_workers=100
)


Found 87140 cached items; annotating 0 uncached items


In [ ]:
import importlib
import hypothesaes.evaluation
importlib.reload(hypothesaes.evaluation)
from hypothesaes.evaluation import score_hypotheses

In [ ]:
test_delta_annots = {key : headline_A_annotations[key] - headline_B_annotations[key] for key in headline_A_annotations.keys()}
metrics, evaluation_df, _ = score_hypotheses(
    hypothesis_annotations=test_delta_annots,
    y_true=labels_holdout,
    classification=True
)


Optimization terminated successfully.
         Current function value: 0.655231
         Iterations 5


In [ ]:
from hypothesaes.evaluation import compute_ols_raw
_, training_model = compute_ols_raw(
    X=Z_delta,
    y_true=train_labels,
    classification=True
)

In [37]:
def display_df(evaluation_df):
    pd.set_option('display.max_colwidth', None)
    display(evaluation_df.sort_values(by='separation_score', ascending=False).round(4))
    pd.reset_option('display.max_colwidth')
    
    print(f"AUC: {metrics['auroc']:.3f}")
    print(f"Significant hypotheses: {metrics['Significant'][0]}/{metrics['Significant'][1]} " 
          f"(p < {metrics['Significant'][2]:.3e})")

In [38]:
display_df(evaluation_df)

,hypothesis,separation_score,separation_pval,regression_coef,regression_pval,feature_prevalence,significant,auc
Index,,,,,,,,
13,includes a reference to a time duration measured in seconds or minutes,0.2085,0.0,0.9647,0.0000,0.0367,True,0.5149
11,"mentions a racial identifier such as the words 'Black', 'White', or the term 'Race'",0.1851,0.0,0.8279,0.0001,0.0275,True,0.5100
3,"includes a reference to visual media such as photos, pictures, images, graphics, or a photographer",0.1751,0.0,0.8466,0.0000,0.0939,True,0.5304
5,"contains a third‑person pronoun (He, They, Their) directly followed by a past‑tense verb (including irregular forms or negative contractions)",0.1569,0.0,0.6757,0.0000,0.0964,True,0.5293
4,contains a standalone female pronoun ('She' or 'Her') as an individual word,0.1545,0.0,0.4779,0.0759,0.0819,False,0.5244
14,"contains the female pronoun \""She\"" or \""Her\",0.1535,0.0,0.2129,0.4198,0.0849,False,0.5250
18,"contains a humor‑related keyword (e.g., “joke”, “funny”, “hilarious”, “comedian”, “comedy”, “funnyman”)"".",0.1434,0.0,0.6690,0.0001,0.0388,True,0.5111
17,"contains the word \""Girl\"" or \""Girls\",0.1351,0.0,0.4854,0.0127,0.0303,False,0.5081
12,"contains the word “Guy”, “Man”, or “Dude”",0.1169,0.0,0.4351,0.0004,0.0709,True,0.5157


AUC: 0.653
Significant hypotheses: 11/20 (p < 5.000e-03)


In [33]:
import sys
evaluation_df.to_csv(sys.stdout, sep=',')

Index,hypothesis,separation_score,separation_pval,regression_coef,regression_pval,feature_prevalence,significant,auc
13,includes a reference to a time duration measured in seconds or minutes,0.20854704731670853,9.690468410702288e-09,0.9647183994898401,9.242720542587357e-08,0.036722515492311224,True,0.5148861524457312
11,"mentions a racial identifier such as the words 'Black', 'White', or the term 'Race'",0.18514837606535395,4.838149577711747e-08,0.8279464372764284,5.5833625254066233e-05,0.027541886619233416,True,0.509971511772465
3,"includes a reference to visual media such as photos, pictures, images, graphics, or a photographer",0.1751115171744737,1.1241457626536867e-30,0.8465502955905014,9.47058291941759e-12,0.09387193022722057,True,0.530416450687131
5,"contains a third‑person pronoun (He, They, Their) directly followed by a past‑tense verb (including irregular forms or negative contractions)",0.15685680418979886,1.7652033918079212e-37,0.6757493754196249,2.2778405860180617e-10,0.096